# C4: Distributed Data Sharding

In Distributed Data Parallel (DDP) training, it is crucial that every worker (rank) receives the exact same number of batches per epoch. If workers receive a different number of batches, they will participate in a different number of `Allreduce` communication steps. This mismatch inevitably leads to deadlocks, as some workers wait for gradient synchronization while others have already finished the epoch.

## Before: The Mathematical Flaw in `Iterators.partition`

The original implementation of `DistributedDataContainer` in Flux.jl partitioned data as follows:

```julia
total_size = numobs(data)
split_across = total_workers(backend)
size_per_worker = Int(ceil(total_size / split_across))

partitions = collect(Iterators.partition(1:total_size, size_per_worker))
idxs = collect(partitions[local_rank(backend) + 1])
```

Consider a dataset of $N=9$ items distributed across $W=4$ workers:
- `size_per_worker = ceil(9 / 4) = 3`
- `Iterators.partition(1:9, 3)` produces exactly 3 partitions:
  1. `1:3` (for Rank 0)
  2. `4:6` (for Rank 1)
  3. `7:9` (for Rank 2)

### Consequences
1. **BoundsError**: When Rank 3 attempts to access the 4th partition (`partitions[4]`), it encounters a `BoundsError` because only 3 partitions exist.
2. **Deadlocks**: Even if $N$ is cleanly divisible by `size_per_worker` but the last chunk is smaller, ranks might produce a different number of batches. For instance, if $N=10$ and $W=4$:
   - `size_per_worker = ceil(10 / 4) = 3`
   - Partitions: `1:3` (Rank 0), `4:6` (Rank 1), `7:9` (Rank 2), `10:10` (Rank 3)
   - Rank 3 gets 1 item. If the batch size is 2 (drop_last=false), Ranks 0, 1, 2 get 2 batches, but Rank 3 gets 1 batch. Rank 3 will finish its epoch early, and Ranks 0, 1, 2 will hang waiting for Rank 3 to join the second `Allreduce`.


## After: PyTorch-Style Dataset Padding

To fix the mathematical flaw and prevent deadlocks, `DistributedDataContainer` now automatically pads the dataset indices to be evenly divisible by the number of workers, mirroring PyTorch's default `DistributedSampler` behavior.

```julia
total_size = numobs(data)
split_across = total_workers(backend)
size_per_worker = Int(ceil(total_size / split_across))

# Pad the dataset size so that it is evenly divisible by the number of workers
total_padded = size_per_worker * split_across
indices = collect(1:total_size)
if total_padded > total_size
    append!(indices, 1:(total_padded - total_size))
end

partitions = collect(Iterators.partition(indices, size_per_worker))
idxs = collect(partitions[local_rank(backend) + 1])
```

With the new logic, given $N=9$ items and $W=4$ workers:
- `size_per_worker = ceil(9 / 4) = 3`
- `total_padded = 3 * 4 = 12`
- We append indices `1:3` to the dataset partitioning indices.
- The `Iterators.partition` produces exactly 4 partitions:
  1. `1:3` (for Rank 0)
  2. `4:6` (for Rank 1)
  3. `7:9` (for Rank 2)
  4. `1:3` (for Rank 3)

### Verification
Every rank now successfully receives a partition. Because `size_per_worker` is identical for all ranks, every rank will process exactly the same number of batches. This guarantees that all ranks will participate in the exact same number of `Allreduce` collective operations during an epoch, cleanly eliminating the risk of mismatched batch counts and preventing DDP deadlocks.


## The Padding Trade-Off: Duplication vs. Deadlocks

You may notice that in our $N=9, W=4$ example, Rank 0 and Rank 3 receive the exact same data indices (`1:3`). In this toy scenario, 33% of our dataset is duplicated. 

This exposes the fundamental mathematical trade-off in distributed training: because we **must** have the exact same number of batches on every rank to prevent `Allreduce` deadlocks, we have to either add fake data (padding) or throw data away (dropping).

Why padding is the industry standard (mirroring PyTorch's default `DistributedSampler`):

1. **Negligible Bias at Scale**: The maximum number of items ever padded is $W - 1$. In a real-world scenario with 1,000,000 images and 8 GPUs, we pad at most 7 images. The statistical bias of seeing 7 images twice out of a million is completely mathematically negligible, whereas a deadlock is catastrophic.
2. **Shuffling Mitigates Overfitting**: In a real training loop, dataset indices are shuffled at the start of every epoch (using an epoch-dependent seed shared across all ranks). The padding happens *after* the shuffle. Therefore, the 1 to $W-1$ items that get duplicated are completely random every epoch, ensuring no single piece of data is systematically over-represented throughout training.
3. **Data Utilization**: The alternative to padding is truncating the dataset (`drop_last`). Padding guarantees that 100% of the dataset is seen during an epoch, even if a tiny fraction is seen twice.
